# Evo2 DNA Sequence Generation and Scoring

![Evo2 DNA Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/evo2/hero.png)

This notebook demonstrates examples of how to use both tools involving the Evo2 model. In the first section, we use `run_evo2_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling with KV caching for efficient generation. In the second section, we use `run_evo2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo2")
display_overview("evo2")
display_docs_section("evo2", "Background")

# Evo2

Evo2 is an autoregressive DNA language model from Arc Institute and Stanford, trained at single-nucleotide resolution across all domains of life. This toolkit wraps it to generate new DNA sequences from a prompt and to score how likely supplied DNA sequences are under the model.

Evo2 ([Brixi et al., 2026](https://doi.org/10.1038/s41586-026-10176-5)) is a DNA language model trained with an autoregressive objective: during training the model learns to predict the next nucleotide given all preceding nucleotides. Training used the [OpenGenome2](https://huggingface.co/datasets/arcinstitute/opengenome2) dataset, which spans bacterial, archaeal, eukaryotic, and phage genomes across all domains of life, so the model is not restricted to any single clade. It is available at several scales, the largest being 40 billion parameters, and uses the StripedHyena 2 architecture, a sequence model that combines convolutional state-space layers with a smaller number of attention layers. This design lets the model process very long stretches of DNA, up to roughly one million nucleotides for the long-context checkpoints, without the memory cost a pure attention model would incur at that length. Several checkpoints are also offered with shorter context windows for lower memory use, and one variant is trained specifically on Microviridae phage genomes.

The autoregressive objective yields two capabilities directly. Sampling from the predicted next-nucleotide distributions produces new candidate sequences, and reading off the probabilities the model assigns to an existing sequence gives a likelihood score that reflects how closely the sequence matches the patterns seen during training. Evo2 is the second model in the Evo family; the earlier [Evo1](https://bio-pro.mintlify.app/tools/causal-models/evo1) was trained only on prokaryotic and phage genomes, whereas Evo2 extends to eukaryotic genomes and longer context.

## Available tools

In [2]:
display_available_tools("evo2")

- **`run_evo2_sample()`** — Sample DNA sequences using Evo2 language model
- **`run_evo2_score()`** — Score DNA sequences using Evo2 language model

### `run_evo2_sample`

This tool utilizes Evo2 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over genomic DNA. The `temperature` and `top_k` parameters control sampling diversity: lower values produce conservative, high-confidence extensions while higher values allow more exploratory generation.

In [3]:
from proto_tools import (
    Evo2SampleInput,
    Evo2SampleConfig,
    run_evo2_sample,
)

In [4]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_sample")

# Seed generation with a real E. coli lacZ coding-sequence fragment.
inputs = Evo2SampleInput(prompts=["ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>prompts</code> | <code>list[str]</code> | required | Prompt sequence(s) to condition generation on |

In [5]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_sample")

# Configure sampling parameters | see docs above for all fields
config = Evo2SampleConfig(
    max_new_tokens=200,
    temperature=0.8,
    top_k=4,
    device="cuda",
)

**Config** — `Evo2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1800</code> | Maximum execution time in seconds |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>prepend_prompt</code> | <code>bool</code> | <code>True</code> | Include the input prompt at the start of each generated sequence |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature for sampling; lower is more deterministic |
| <code>top_p</code> | <code>float</code> | <code>1.0</code> | Nucleus sampling cutoff over per-position token probabilities |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Prompts per GPU forward pass; raise for throughput, lower if OOM |
| <code>model_checkpoint</code> | <code>Literal['evo2_7b', 'evo2_20b', 'evo2_40b', 'evo2_7b_base', 'evo2_40b_base', 'evo2_1b_base', 'evo2_7b_262k', 'evo2_7b_microviridae']</code> | <code>'evo2_7b'</code> | Evo2 weights variant |
| <code>local_path</code> | <code>str &#124; None</code> | <code>None</code> | Override the default download with a local weights directory |
| <code>top_k</code> | <code>int</code> | <code>4</code> | Limit sampling to the top-k most probable tokens at each step |
| <code>max_new_tokens</code> | <code>int</code> | <code>32</code> | Maximum newly-generated tokens per prompt (excludes the prompt) |
| <code>cached_generation</code> | <code>bool</code> | <code>True</code> | Use the model's per-call KV cache during generation |
| <code>force_prompt_threshold</code> | <code>int &#124; None</code> | <code>None</code> | Tokens to prefill in parallel before switching to prompt forcing |
| <code>max_seqlen</code> | <code>int &#124; None</code> | <code>None</code> | Maximum sequence length the KV cache will be sized for |
| <code>skip_special_tokens</code> | <code>bool</code> | <code>False</code> | Filter EOS/PAD bytes from the detokenized output |
| <code>stop_at_eos</code> | <code>bool</code> | <code>True</code> | Stop generation when an EOS (id=0) token is sampled |
| <code>old_kv_cache</code> | <code>Evo2KVCacheRef &#124; None</code> | <code>None</code> | Worker-local KV cache handle to use for continued generation |
| <code>return_kv_cache</code> | <code>bool</code> | <code>False</code> | Return worker-local KV cache handles for continued generation |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |

In [6]:
# Run the sampling function
sample_result = run_evo2_sample(inputs, config)

Running run_evo2_sample [00:00]

In [7]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")

**Output** — `Evo2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Generated sequences |
| <code>logits</code> | <code>list[Any] &#124; None</code> | <code>None</code> | Per-step logits for each generated sequence (shape [n_outputs, gen_len, vocab_size]) |
| <code>kv_caches</code> | <code>list[Evo2KVCacheRef] &#124; None</code> | <code>None</code> | Opaque worker-local KV cache handles (only valid on the same worker) |

**`Evo2KVCacheRef`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>type</code> | <code>Literal['evo2_kv_cache']</code> | <code>'evo2_kv_cache'</code> | Evo2 KV-cache handle type |
| <code>cache_id</code> | <code>str</code> | required | Identifier for the worker-local KV cache |

Prompt:       ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG
Generated:    ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGGACTTTGCAGTGGGTTCATGTCGCCGATTC...
Total length: 251 nucleotides


### `run_evo2_score`

Evo2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each nucleotide is given its preceding context. The resulting perplexity provides a single-number summary of sequence naturalness: lower perplexity means the sequence more closely matches the statistical patterns in the training data. This is useful for ranking generated candidates, comparing wild-type and engineered variants, or filtering out sequences that deviate substantially from natural genomic patterns.

In [8]:
from proto_tools import (
    Evo2ScoringInput,
    Evo2ScoringConfig,
    run_evo2_score,
)

In [9]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_score")

# Contrast a natural coding sequence with a shuffled control of identical composition;
# Evo2 should assign the natural sequence the lower (better) perplexity.
score_inputs = Evo2ScoringInput(sequences=["ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG", "GACGTTCATGCACTAGGTTGATCTGTCCGATAAACTGGCATGCCTTAGCTG"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Sequences to score |

In [10]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_score")

# Configure scoring | see docs above for all fields
score_config = Evo2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",
)

**Config** — `Evo2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1800</code> | Maximum execution time in seconds |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>model_checkpoint</code> | <code>Literal['evo2_7b', 'evo2_20b', 'evo2_40b', 'evo2_7b_base', 'evo2_40b_base', 'evo2_1b_base', 'evo2_7b_262k', 'evo2_7b_microviridae']</code> | <code>'evo2_7b'</code> | Evo2 weights variant |
| <code>local_path</code> | <code>str &#124; None</code> | <code>None</code> | Override HuggingFace download with a local weights directory |
| <code>prepend_bos</code> | <code>bool</code> | <code>False</code> | Prepend a beginning-of-sequence token before scoring |

In [11]:
# Run the scoring function
score_result = run_evo2_score(score_inputs, score_config)

Running run_evo2_score [00:00]

In [12]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[CausalModelScoringMetrics]</code> | required | List of scoring outputs, one per input sequence |

**Metrics** — `CausalModelScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  |  |

Sequence 1: ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG
    Log-likelihood:     -70.169
    Avg log-likelihood: -1.403
    Perplexity:         4.069
Sequence 2: GACGTTCATGCACTAGGTTGATCTGTCCGATAAACTGGCATGCCTTAGCTG
    Log-likelihood:     -70.957
    Avg log-likelihood: -1.419
    Perplexity:         4.134


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo2_sequences.fasta'}")

score_result.export(name="evo2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo2_scores.csv'}")

Generated sequences exported to example_output/evo2_sequences.fasta
Scores exported to example_output/evo2_scores.csv
